In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import fnmatch
from connect import bob
from utils import *
from limb_fitting import *
from fit_cld import *
from scipy.ndimage import gaussian_filter

In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']

In [3]:
sftp = bob()

top_dir = '/data/slam/valori/test_l2_fmdb/FDT_test_release_v08_2025/v4/l2/'
#top_dir = '/data/solo/phi/data/fmdb/l1/'
dirs = sorted(sftp.listdir(top_dir))

Q = []

output_file = 'cld_fit.csv'

with open(output_file, 'w') as f:
    f.write('date, did, alpha, beta, epsilon, scale, bias, drsun, sigma\n')

for directory in dirs:
    #if fnmatch.fnmatch(directory, '2024*') or fnmatch.fnmatch(directory, '2025*'):
    if fnmatch.fnmatch(directory, '2024-01*'):
        for file in sorted(sftp.listdir(top_dir + directory)):
            if fnmatch.fnmatch(file, '*stokes*.fits.gz'):
                try:
                    print(file)

                    remote_file = top_dir + directory + '/' + file
                    local_file = 'temp.fits.gz'
                    sftp.get(remote_file, local_file)

                    with fits.open(local_file) as hdul:
                        header = hdul[0].header
                        data = hdul[0].data

                    date = file.split('/')[-1].split('_')[3]
                    did = file.split('/')[-1].split('_')[-1].split('.')[0]

                    cpos = header['CONTPOS'] - 1
                    xr, yr = reflection_point_predict(header)

                    image = data[cpos,0].copy()
                    image = undistort(image, header, xd, yd)
                    xc, yc, rsun = find_center(image)

                    phi = np.arctan((yc - yr) / (xc - xr)) * 180 / np.pi

                    params, r, q, q_ = fit_cld(image, phi0=phi-90, phi1=phi+90)
                    alpha, beta, epsilon, scale, bias, rsun_, sigma = params

                    drsun = rsun_ - rsun

                    with open(output_file, 'a') as f:
                        f.write(f'{date}, {did}, {alpha:.4f}, {beta:.4f}, {epsilon:.4f}, {scale:.4f}, {bias:.4f}, {drsun:.4f}, {sigma:.4f}\n')

                   # print(params, rsun)

                except:
                    pass

                stop

solo_L2_phi-fdt-stokes_20240101T040003_V202602220902_0441010503.fits.gz


NameError: name 'stop' is not defined

In [4]:
plt.figure(figsize=(10,10))
plt.plot(r, q)
plt.plot(r, q_)

plt.xlim(rsun-20, rsun+200)
plt.ylim(-0.01,0.01)
plt.grid(True)
plt.tight_layout()